In [1]:
import numpy as np
import pandas as pd
import Soil_Carbon_Model as md
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit 
import os
import openpyxl
from pandas import *
%matplotlib inline

In [2]:
import pylab
params = {'backend': 'ps',
          'axes.labelsize': 20,
          'legend.fontsize': 20,
          'legend.handlelength': 2.0,
          'legend.borderaxespad': 0,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'font.family': 'serif',
          'font.size': 20,
          'font.weight': 'normal',
          #'font.serif': ['Palatino'],
          'ps.usedistiller': 'xpdf',
          'text.usetex': False,
         'agg.path.chunksize' : 1000}
pylab.rcParams.update(params)

In [3]:
dt = 1 / 24
nyears = 2000
ndays = nyears * 365
nsim = ndays * 24
nnpp = ndays * 24

#Rainfall from Poisson
rain = md.poisson_process(n = int(nsim), alpha = 16.64/1000, lbd = (0.24 * dt))

tsim = np.linspace(1, nnpp, int(nnpp)) # Hour

days = np.arange(365)
annual_avg_npp = 996.5

# Half sine wave shape over 365 days
shape = np.sin((np.pi*days)/365)                       # sinB(x-h) + K; B = pi/365, h = 0, K = 0

# Normalize so that the sum equals the annual NPP
daily_npp = annual_avg_npp * (shape / np.sum(shape))
hourly_npp = np.repeat(daily_npp, 24)

npp = np.tile(hourly_npp,nyears)

In [4]:
print("Maximum Value is:", max(npp))
print("Length is:", len(npp))
print("Average value is:", np.mean(npp))

Maximum Value is: 4.288475912207284
Length is: 17520000
Average value is: 2.730136986301364


In [5]:
total_rainfall = rain.sum()
print(f"Total rainfall over 450 years = {total_rainfall:.3f} meters")

RPY = (total_rainfall / nyears)*1000         # average rainfall per year
print(f"Average rainfall per year = {RPY:.3f} mm/year")

Total rainfall over 450 years = 2925.340 meters
Average rainfall per year = 1462.670 mm/year


In [6]:
PET = 1119.8
AI = PET/RPY
print(AI)

"""
def hill_constrained(R, KA, n):
    R_max = 4.18

    numerator = R**n
    denominator = KA**n + R**n
    norm_factor = R_max**n / (KA**n + R_max**n)
    TI = (numerator / denominator) / norm_factor

    # Clip to ensure TI does not exceed 1
    return (np.minimum(TI, 1.0))

TI = hill_constrained(AI, 3.45, 2.28)
print(TI)
"""

# Final best-fit parameters from your latest run
KA_final = 9.0132
n_final = 1.7444
R_max = 3.8  # Max AI used for normalization

def compute_TI(AI):
    AI = np.asarray(AI)  # Accept scalars or arrays
    numerator = AI**n_final
    denominator = KA_final**n_final + AI**n_final
    norm_factor = R_max**n_final / (KA_final**n_final + R_max**n_final)
    TI = (numerator / denominator) / norm_factor
    return np.minimum(TI, 1.0)

TI = compute_TI(AI)
print("TI =", TI)

0.7655862075587948
TI = 0.07368331822334387


In [7]:
#Hydraulic parameterization for dynamic porosity -------------------------------
alpha1 = 580  #hydraulic conductivity shape parameter
beta1 = 6     #hydraulic conductivity shape parameter
alpha2 = 580  #porosity shape parameter
beta2 = 6     #porosity shape parameter
kstex = 0.1   #m/d,
kstot = 1     #m/d
phitex = 0.50  
phitot = 0.60  

In [8]:
# Simulation for 2150 years at dynamic porosity
sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, alpha1, beta1, alpha2, beta2, TI)
sim_year = sim.pivot_table(values=['s', 'phi', 'ksstr', 'P', 'DOC', 'A', 'M', 'B',
                                   'Fa', 'Fpa','Flb','Fl', 'Flm', 'Fpl', 'Fbm','Fld', 'Fal',
                                   'lk','npp_', 'L','Fi', 'Flpom', 'decomp'],
                           index='Years')

print(sim_year)

100%|██████████| 17520000/17520000 [27:52<00:00, 10473.59it/s]


                 A           B         DOC        Fa       Fal       Fbm  \
Years                                                                      
1       942.518700   74.097868  135.125591  0.251138  0.006987  0.543558   
2      1132.798256   54.718749  230.668303  0.311526  0.008058  0.401341   
3      1297.045858   93.459987  253.521373  0.341721  0.010110  0.685477   
4      1438.835489  119.624152  221.299088  0.400997  0.012431  0.877414   
5      1534.848878  131.768974  209.237927  0.428467  0.013486  0.966503   
...            ...         ...         ...       ...       ...       ...   
1996   1780.382406  163.562638  202.090956  0.496937  0.016106  1.199714   
1997   1779.925975  161.861793  205.768877  0.493870  0.015984  1.187234   
1998   1779.374431  164.992223  207.317872  0.507321  0.016460  1.210183   
1999   1778.335080  159.903780  214.925483  0.479603  0.015493  1.172885   
2000   1779.205817  160.189077  220.753077  0.466877  0.015089  1.174956   

           

In [11]:
sim_year.to_csv(r"C:\Users\rishabh.singh.AGNET\OneDrive - Texas A&M AgriLife\Desktop\Slope\Zanatta\ZAN.csv", index= True, header = True)

sim_year.to_csv(r"C:\Users\rishabh.singh.AGNET\OneDrive - Texas A&M AgriLife\Desktop\Slope\Results\ZAN.csv", index= True, header = True)